# Notebook 01 — What is Machine Learning and the ML Pipeline

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 01 of 15  
**Time estimate:** 60 minutes

> ML is not magic. It is optimization on a loss function over a function class,
> evaluated on held-out data. Everything else — the models, the tricks, the
> interview questions — is a footnote to this sentence.

---
## Step 1 — Motivation

Biology generates high-dimensional data faster than we can form hypotheses:
50,000-gene expression profiles, billions of genetic variants, millions of cell images.
Machine learning provides systematic tools to find patterns in this data — to build
predictive models when the underlying rules are too complex to write by hand.

---
## Step 2 — Intuition

**Supervised learning:** Given labeled examples $(\mathbf{x}_i, y_i)$, learn a
function $f: \mathcal{X} \to \mathcal{Y}$ that generalizes to new inputs.
- **Classification:** $y \in \{0,1,...,K\}$ — cancer/no-cancer, cell type, variant effect
- **Regression:** $y \in \mathbb{R}$ — gene expression, drug IC50, protein binding affinity

**Unsupervised learning:** No labels — find structure in $\{\mathbf{x}_i\}$.
- Clustering: patient stratification, cell type discovery
- Dimensionality reduction: visualization, feature compression

**The core tension — bias-variance tradeoff:**
- **High bias (underfitting):** model too simple to capture the pattern
- **High variance (overfitting):** model memorizes training data, fails on test data
- The sweet spot depends on dataset size, noise level, and problem complexity

---
## Step 3 — Biological Background

**Biology-specific ML challenges:**

1. **High dimensional, small-n (p >> n):** RNA-seq: 20,000 genes, 50 samples.
   Classical statistics breaks down; regularization is mandatory.

2. **Class imbalance:** Rare diseases, rare variants, rare cell types. A model that
   always predicts "benign" achieves 99.9% accuracy on a 0.1%-prevalence disease —
   this is useless. Always check class balance before any ML.

3. **Batch effects:** Technical variation between experiments (lab, day, protocol)
   can be larger than biological signal. A model trained on Batch A and tested on
   Batch B will fail — not because the biology changed, but because the tech did.

4. **Data leakage:** The most common mistake in biological ML.
   - Feature selection on all data before splitting → inflated performance
   - Patient-level correlation not respected (multiple samples per patient in train AND test)
   - Using future information (e.g., selecting features that correlate with the outcome
     using test data)
   Rule: the test set must be completely invisible until final evaluation.

5. **Interpretability:** "Black box" models can achieve high accuracy but are often
   not accepted by biologists or clinicians. Feature importance and model
   explainability are often as important as predictive performance.

6. **Reproducibility:** Random seeds, package versions, and data preprocessing order
   all affect results. Set seeds, log versions, and record every preprocessing step.

---
## Step 4 — Mathematical Explanation

**Bias-variance decomposition** (for squared error):
$$\mathbb{E}[(y - \hat{f}(x))^2] = \underbrace{(\mathbb{E}[\hat{f}(x)] - f(x))^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}[\hat{f}(x) - \mathbb{E}[\hat{f}(x)]]^2}_{\text{Variance}} + \sigma^2$$

- $\text{Bias}^2$: systematic error (model class too simple)
- $\text{Variance}$: sensitivity to training data fluctuations (model too complex)
- $\sigma^2$: irreducible noise

**No Free Lunch Theorem (Wolpert 1996):**
No single algorithm is best for all problems. Averaged over all possible data
distributions, every algorithm performs equally. Choosing an algorithm is a choice
to impose prior knowledge about your specific problem structure.

**The standard ML pipeline:**
1. Problem formulation → choose supervised/unsupervised, classification/regression, metric
2. Data collection + cleaning → missing values, outliers, duplicates
3. Exploratory data analysis (EDA) → distributions, correlations, class balance
4. Feature engineering → domain-specific transformations
5. Train/validation/test split → before feature selection (leakage prevention)
6. Model selection + hyperparameter tuning → CV on train+validation only
7. Final evaluation → test set, once, report honestly
8. Interpretation + deployment → feature importance, calibration

In [ ]:
# Step 6 — Python: Bias-variance tradeoff demonstration
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

# True function: sin(2*pi*x) with Gaussian noise
rng = np.random.default_rng(42)

def true_f(x):
    return np.sin(2 * np.pi * x)

def generate_data(n, noise=0.3, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, n)
    y = true_f(x) + rng.normal(0, noise, n)
    return x.reshape(-1,1), y

x_test, y_test = generate_data(200, seed=99)
x_plot = np.linspace(0, 1, 200).reshape(-1,1)

# Compare degrees 1, 3, 15 on 20 bootstrap datasets
degrees = [1, 3, 15]
n_bootstrap = 50
results = {}

for deg in degrees:
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    preds = []
    for bs in range(n_bootstrap):
        x_tr, y_tr = generate_data(20, seed=bs)
        model.fit(x_tr, y_tr)
        preds.append(model.predict(x_plot))
    preds = np.array(preds)  # shape: (n_bootstrap, n_plot)
    mean_pred = preds.mean(axis=0)
    bias_sq = (mean_pred - true_f(x_plot.ravel()))**2
    variance = preds.var(axis=0)
    # Also compute test MSE on one training set
    x_tr, y_tr = generate_data(20, seed=0)
    model.fit(x_tr, y_tr)
    train_mse = np.mean((model.predict(x_tr) - y_tr)**2)
    test_mse = np.mean((model.predict(x_test) - y_test)**2)
    results[deg] = dict(preds=preds, mean=mean_pred, bias_sq=bias_sq,
                          variance=variance, train_mse=train_mse, test_mse=test_mse)
    print(f'Degree {deg:2d}: train MSE={train_mse:.4f}, test MSE={test_mse:.4f}, '
            f'mean bias²={bias_sq.mean():.4f}, mean var={variance.mean():.4f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for col, deg in enumerate(degrees):
    r = results[deg]
    
    # Top row: bootstrap predictions
    ax = axes[0, col]
    for pred in r['preds'][:20]:
        ax.plot(x_plot.ravel(), pred, 'steelblue', alpha=0.15, lw=0.8)
    ax.plot(x_plot.ravel(), r['mean'], 'steelblue', lw=2, label='Mean prediction')
    ax.plot(x_plot.ravel(), true_f(x_plot.ravel()), 'k', lw=2, label='True function')
    ax.set_title(f'Degree {deg} polynomial\ntest MSE={r["test_mse"]:.3f}', fontsize=9)
    ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=7)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    
    # Bottom row: bias² and variance
    ax = axes[1, col]
    ax.fill_between(x_plot.ravel(), 0, r['bias_sq'], alpha=0.5,
                      color='tomato', label=f'Bias² (mean={r["bias_sq"].mean():.3f})')
    ax.fill_between(x_plot.ravel(), r['bias_sq'], r['bias_sq'] + r['variance'],
                      alpha=0.5, color='steelblue', label=f'Variance (mean={r["variance"].mean():.3f})')
    ax.set_xlabel('x')
    ax.set_ylabel('Error')
    ax.set_title(f'Bias²+Variance decomposition')
    ax.legend(fontsize=7)

plt.suptitle('Module 13 NB01: Bias-Variance Tradeoff\n(20 training points, 50 bootstrap samples)',
               fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('bias_variance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Demonstrate data leakage: feature selection BEFORE vs. AFTER split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Generate pure noise dataset: 50 samples, 500 features, random binary labels
# No true signal — accuracy should be ~50%
rng = np.random.default_rng(0)
X_noise = rng.standard_normal((50, 500))
y_noise = rng.integers(0, 2, 50)

# WRONG: Select top-20 features on ALL data, THEN cross-validate
selector = SelectKBest(f_classif, k=20)
X_selected_wrong = selector.fit_transform(X_noise, y_noise)  # leakage!
acc_wrong = cross_val_score(LogisticRegression(max_iter=1000), X_selected_wrong,
                              y_noise, cv=5, scoring='accuracy').mean()

# CORRECT: Feature selection INSIDE each CV fold (use sklearn Pipeline)
from sklearn.pipeline import Pipeline
pipe = Pipeline([('select', SelectKBest(f_classif, k=20)),
                   ('clf', LogisticRegression(max_iter=1000))])
acc_correct = cross_val_score(pipe, X_noise, y_noise, cv=5, scoring='accuracy').mean()

print('=== DATA LEAKAGE DEMONSTRATION ===')
print(f'Dataset: 50 samples, 500 RANDOM features, random labels (no signal)')
print(f'Expected accuracy: ~0.50')
print(f'WRONG (leaky feature selection): {acc_wrong:.3f}  ← inflated by leakage')
print(f'CORRECT (pipeline): {acc_correct:.3f}  ← honest estimate')
print()
print('Lesson: always put feature selection INSIDE the cross-validation loop.')

---
## Step 8 — Exercises

1. Run the bias-variance experiment with `n=5` training points instead of 20.
   What happens to variance for the degree-15 model?
2. Extend the leakage demonstration: what happens if you use 1000 features instead
   of 500? Does the leakage-inflated accuracy increase?
3. For a dataset with 1000 cancer samples and 100 benign samples, which evaluation
   metric would you report and why?
4. Describe a realistic scenario where batch effects in RNA-seq data would cause a
   model to appear to perform well but fail on new patient data.

---
## Step 10 — Quiz

1. What is the bias-variance tradeoff? Which increases as model complexity increases?
2. What is overfitting? Give one symptom you would see in training vs. test performance.
3. What is data leakage in the context of ML? Give a biology-specific example.
4. What does the No Free Lunch Theorem say?
5. List the 8 steps of the ML pipeline in order.

---
## Step 12 — Reflection

> *[In one paragraph: explain why the leakage demonstration showed ~70% accuracy on
> a random dataset. What is being 'learned' when feature selection happens before
> cross-validation?]*

---
*Next: `02_linear_regression_from_scratch.ipynb`*